# Problem 1: RNN, LSTM, and GRU Sequence Models


In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Set seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)



## 1. Data Pipeline & Adding Problem Verification


In [1]:
# Dataset 3: Adding Problem Generator
def make_adding(n, T, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)

# Unit test for adding problem dataset generator
x_test, y_test = make_adding(100, 50, seed=42)
assert x_test.shape == (100, 50, 2)
assert y_test.shape == (100,)
for i in range(100):
    idx1 = np.where(x_test[i, :25, 1] == 1)[0]
    idx2 = np.where(x_test[i, 25:, 1] == 1)[0]
    assert len(idx1) == 1 and len(idx2) == 1
    expected_sum = x_test[i, idx1[0], 0] + x_test[i, 25 + idx2[0], 0]
    np.testing.assert_allclose(y_test[i], expected_sum, rtol=1e-5)
print("Adding problem unit tests PASSED successfully!")



## 2. Cells from Scratch & Parameter Reconciliations


In [1]:
# Task 2: Cells From Scratch
class MyRNNCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(hidden_size, input_size) / math.sqrt(hidden_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) / math.sqrt(hidden_size))
        self.b_ih = nn.Parameter(torch.zeros(hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor = None):
        if h_prev is None:
            h_prev = torch.zeros(x.size(0), self.hidden_size, device=x.device)
        h_next = torch.tanh(x @ self.W_ih.T + self.b_ih + h_prev @ self.W_hh.T + self.b_hh)
        return h_next

class MyLSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, forget_bias_init: float = 1.0):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(4 * hidden_size, input_size) / math.sqrt(hidden_size))
        self.W_hh = nn.Parameter(torch.randn(4 * hidden_size, hidden_size) / math.sqrt(hidden_size))
        self.b_ih = nn.Parameter(torch.zeros(4 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(4 * hidden_size))
        
        if forget_bias_init != 0:
            nn.init.constant_(self.b_ih[hidden_size:2*hidden_size], forget_bias_init)

    def forward(self, x: torch.Tensor, states=None):
        if states is None:
            h_prev = torch.zeros(x.size(0), self.hidden_size, device=x.device)
            c_prev = torch.zeros(x.size(0), self.hidden_size, device=x.device)
        else:
            h_prev, c_prev = states

        gates = (x @ self.W_ih.T + self.b_ih) + (h_prev @ self.W_hh.T + self.b_hh)
        i, f, g, o = gates.chunk(4, dim=-1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)

        c_next = f * c_prev + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class MyGRUCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(3 * hidden_size, input_size) / math.sqrt(hidden_size))
        self.W_hh = nn.Parameter(torch.randn(3 * hidden_size, hidden_size) / math.sqrt(hidden_size))
        self.b_ih = nn.Parameter(torch.zeros(3 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(3 * hidden_size))

    def forward(self, x: torch.Tensor, h_prev=None):
        if h_prev is None:
            h_prev = torch.zeros(x.size(0), self.hidden_size, device=x.device)

        gi = x @ self.W_ih.T + self.b_ih
        gh = h_prev @ self.W_hh.T + self.b_hh

        i_r, i_z, i_n = gi.chunk(3, dim=-1)
        h_r, h_z, h_n = gh.chunk(3, dim=-1)

        r = torch.sigmoid(i_r + h_r)
        z = torch.sigmoid(i_z + h_z)
        n = torch.tanh(i_n + r * h_n)

        h_next = (1 - z) * n + z * h_prev
        return h_next

# Verification unit tests matching nn.RNN, nn.LSTM, nn.GRU within 1e-5
x_dummy = torch.randn(16, 9)
h_dummy = torch.randn(16, 64)
c_dummy = torch.randn(16, 64)

# Test RNN
my_rnn = MyRNNCell(9, 64)
torch_rnn = nn.RNNCell(9, 64)
my_rnn.W_ih.data.copy_(torch_rnn.weight_ih)
my_rnn.W_hh.data.copy_(torch_rnn.weight_hh)
my_rnn.b_ih.data.copy_(torch_rnn.bias_ih)
my_rnn.b_hh.data.copy_(torch_rnn.bias_hh)
out_my_rnn = my_rnn(x_dummy, h_dummy)
out_torch_rnn = torch_rnn(x_dummy, h_dummy)
assert torch.allclose(out_my_rnn, out_torch_rnn, atol=1e-5)

# Test LSTM
my_lstm = MyLSTMCell(9, 64)
torch_lstm = nn.LSTMCell(9, 64)
my_lstm.W_ih.data.copy_(torch_lstm.weight_ih)
my_lstm.W_hh.data.copy_(torch_lstm.weight_hh)
my_lstm.b_ih.data.copy_(torch_lstm.bias_ih)
my_lstm.b_hh.data.copy_(torch_lstm.bias_hh)
out_my_h, out_my_c = my_lstm(x_dummy, (h_dummy, c_dummy))
out_torch_h, out_torch_c = torch_lstm(x_dummy, (h_dummy, c_dummy))
assert torch.allclose(out_my_h, out_torch_h, atol=1e-5)
assert torch.allclose(out_my_c, out_torch_c, atol=1e-5)

# Test GRU
my_gru = MyGRUCell(9, 64)
torch_gru = nn.GRUCell(9, 64)
my_gru.W_ih.data.copy_(torch_gru.weight_ih)
my_gru.W_hh.data.copy_(torch_gru.weight_hh)
my_gru.b_ih.data.copy_(torch_gru.bias_ih)
my_gru.b_hh.data.copy_(torch_gru.bias_hh)
out_my_gru = my_gru(x_dummy, h_dummy)
out_torch_gru = torch_gru(x_dummy, h_dummy)
assert torch.allclose(out_my_gru, out_torch_gru, atol=1e-5)

print("All Custom Cell Unit Tests PASSED with numerical precision < 1e-5!")

